# STRIDER tutorial

STRIDER identifies transients from their Roman prism spectra. For each object, it returns the probability of each
class and the redshift with its uncertainty.

This notebook classifies one transient, then adds a redshift prior and a class restriction, reads the flags, and
classifies a batch.

It uses the bundled test model, so it runs without a download; its results are placeholders. For real results, use
`load_model()`, which downloads the released model the first time.

In [1]:
import strider

model = strider.load_model("test")
model.classes

['Ia',
 '91bg',
 'Iax',
 'IIP',
 'IIL',
 'IIb',
 'IIn',
 'Ib',
 'Ic',
 'Ic-BL',
 'SLSN',
 'TDE',
 'ILOT',
 'KN',
 'PISN']

## One transient

The bundled example is a simulated SN Ia at redshift 0.87 with eleven visits. A visit is one prism spectrum. For each pixel it has the
observer-frame wavelength in Å, the flux per unit wavelength in any units, and the 1σ error in the same units. Each
visit also has its date, as an MJD.

In [2]:
transient = strider.load_example()
visit = transient.visits[0]
print(len(transient.visits), "visits; the first on MJD", visit.mjd)
print("wavelength", visit.wavelength[:3])
print("flux      ", visit.flux[:3])
print("error     ", visit.error[:3])

11 visits; the first on MJD 61929.4881
wavelength [7450.55004883 7469.75       7489.09985352]
flux       [ 4.35076447e-18 -9.85374045e-19  3.76069943e-20]
error      [1.91722400e-17 5.17417975e-18 1.72152478e-18]


In [3]:
result = model.classify(transient)
print(result)

STRIDER · transient                                  strider-synthetic-test-1.0
Class     IIP 33% · IIb 15% · Ic 12%
Redshift  0.22 (12% within 0.1) · 68% range 0.35–2.38
Match     low strength
Data      11 visits over 100 days · S/N 4.8
Checks    weak source (60%)


The card gives the visits, the three most likely classes with the redshift each one gives, the redshift with
its peaks and 68% range, and the checks. `result.plot()` draws the figure.

Each number is also a field of the result. To select SN Ia, use `p_ia`.

In [4]:
print("P(Ia):", result.p_ia)
print("most likely:", result.top_class, result.top_probability)
print("redshift:", result.redshift, "68%:", result.redshift_68, "95%:", result.redshift_95)

P(Ia): 0.002723278782085965
most likely: IIP 0.3345803567132607
redshift: 0.2150232412111863 68%: (0.34692046020706785, 2.3816961544281527) 95%: (0.13431252539157867, 3.0)


The redshift above is summed over all classes. If you treat the transient as a SN Ia, use the redshift
given that class:

In [5]:
ia = result.redshift_given("Ia")
print("redshift if SN Ia:", ia.redshift, "68%:", ia.redshift_68)

redshift if SN Ia: 3.0 68%: (0.9705832771774378, 2.88312132493567)


## Your own transient

Put it in a CSV file with one row per pixel and the columns `mjd`, `wavelength`, `flux` and `error`, and optionally
`mask` (True for pixels to ignore). The rows with one MJD are one visit. Then:

```python
from strider.io import read_transient

result = model.classify(read_transient("my_transient.csv"))
```

A pandas or astropy table with the same columns works too: `model.classify(table)`.

## A redshift prior

If you know the host galaxy's spectroscopic or photometric redshift, give it with its 1σ error. The redshift posterior then follows it. The test
model has only 12 redshift cells, so here the redshift can only move to the nearest of them.

In [6]:
hosted = model.classify(transient, host_redshift=0.98, host_redshift_error=0.01)
print(hosted)

STRIDER · transient                                  strider-synthetic-test-1.0
Class     IIP 33% · IIb 14% · Ic 13%
Redshift  0.88 (82% within 0.1) · 68% range 0.72–1.06
Match     low strength
Data      11 visits over 100 days · S/N 4.8
Checks    weak source (60%)


## A class restriction

If you know the transient is one of a few classes, give them. They share the probability.

In [7]:
kept = model.classify(transient, classes=["Ia", "Iax", "91bg"])
print(kept)

STRIDER · transient                                  strider-synthetic-test-1.0
Class     Iax 70% · 91bg 19% · Ia 11%
Redshift  0.18 (12% within 0.1) · 68% range 0.33–2.44
Match     low strength
Data      11 visits over 100 days · S/N 4.8
Checks    weak source (60%)


The probabilities are calibrated with the default prior only. A restriction, like a redshift prior, is
applied exactly, but the calibration is not checked with it.

## The flags

A raised flag means the result needs care:

- `weak_source`: there is probably no transient in the spectra.
- `several_redshift_peaks`: the redshift has more than one likely value.
- `few_classes_trained`: few classes were trained at this redshift (below z 0.5 only Ia, Iax and KN), so the others
  cannot be ruled out.

The second bundled example raises one of them.

In [8]:
batch = strider.load_example(batch=True)
noisy = model.classify(batch["masked_and_bad_pixels"])
print(noisy)
print("flags:", noisy.flags)
print("probability of a transient:", noisy.source_probability)

STRIDER · transient                                  strider-synthetic-test-1.0
Class     Ia 26% · IIP 25% · IIb 11%
Redshift  0.44 (11% within 0.1) · 68% range 0.33–2.36
Match     low strength
Data      2 visits over 25 days · S/N 9.7
Checks    weak source (60%)
flags: {'several_redshift_peaks': False, 'weak_source': True, 'untrained_subtypes': False, 'few_classes_trained': False, 'brightness_failed': False}
probability of a transient: 0.5995790476498807


## A batch

`classify_many` classifies many transients, by name. If one fails, its result is the reason, and the rest carry on.

In [9]:
results = model.classify_many(batch)
for name, found in results.items():
    print(found.card(name), end="\n\n")

STRIDER · sn_ia                                      strider-synthetic-test-1.0
Class     IIP 33% · IIb 15% · Ic 12%
Redshift  0.22 (12% within 0.1) · 68% range 0.35–2.38
Match     low strength
Data      11 visits over 100 days · S/N 4.8
Checks    weak source (60%)

STRIDER · masked_and_bad_pixels                      strider-synthetic-test-1.0
Class     Ia 26% · IIP 25% · IIb 11%
Redshift  0.44 (11% within 0.1) · 68% range 0.33–2.36
Match     low strength
Data      2 visits over 25 days · S/N 9.7
Checks    weak source (60%)

STRIDER · a_masked_block                             strider-synthetic-test-1.0
Class     Ia 36% · IIP 22% · IIb 9%
Redshift  0.76 (11% within 0.1) · 68% range 0.41–2.38
Match     low strength
Data      2 visits over 3 days · S/N 0.2



To keep the results, give a file name: `model.classify_many(batch, "results.csv")`. The ending sets
the format: `.csv` for one row per transient, `.parquet` for the same with the redshift posterior, `.json` for every
detail.

The [documentation](https://roman-supernova-pit.github.io/strider/) explains every result and option.